# Data Exploration — live database

Ad-hoc SQL against the local Postgres (`make postgres-start`), plus a findings log
so what you learn outlives the kernel session.

- **Kernel:** must be the poetry venv, or `from config import ...` won't resolve.
- **Read-only:** `q()` opens a plain connection and never commits. Keep it that way —
  see the "no DB writes uninvited" rule.
- **Findings:** `note(...)` appends to `data/exploration_notes.md`.

In [1]:
import pandas as pd
from sqlalchemy import text

from config import settings, FilesLocationConstants
from db.async_engine import get_async_engine

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)

engine = get_async_engine(settings.sqlalchemy)


async def q(sql: str, **params) -> pd.DataFrame:
    """Run a read-only SELECT and hand back the rows as a DataFrame.

    Bind values instead of f-strings:  await q("... WHERE year > :y", y=2000)
    """
    async with engine.connect() as conn:
        result = await conn.execute(text(sql), params)
        cols = list(result.keys())
        return pd.DataFrame(result.fetchall(), columns=cols)


print(f"connected to {settings.sqlalchemy.HOST}:{settings.sqlalchemy.PORT}/{settings.sqlalchemy.DB}")

connected to localhost:5432/book_recommender


In [2]:
# sanity check — exact row counts (pg_stat_user_tables reads 0 until ANALYZE runs)
await q("""
    SELECT 'books' AS table, count(*) AS rows FROM books
    UNION ALL SELECT 'chat_runs', count(*) FROM chat_runs
    UNION ALL SELECT 'test_runs', count(*) FROM test_runs
    UNION ALL SELECT 'feedback',  count(*) FROM feedback
    ORDER BY rows DESC
""")

,table,rows
0,books,5197
1,chat_runs,34
2,test_runs,0
3,feedback,0


## Queries

Each query gets its own cell. Note what you find below it so the next session
doesn't have to re-derive it.

In [3]:
# how lumpy is `categories`?
cats = await q("""
    SELECT categories, count(*) AS n
    FROM books
    GROUP BY categories
    ORDER BY n DESC
""")
print(f"{len(cats)} distinct categories over {cats['n'].sum()} books")
cats.head(25)

480 distinct categories over 5197 books


,categories,n
0,Fiction,2111
1,Juvenile Fiction,390
2,Biography & Autobiography,311
3,History,207
4,Literary Criticism,124
5,Religion,117
6,Philosophy,117
7,Comics & Graphic Novels,116
8,Drama,86
9,Juvenile Nonfiction,57


In [4]:
# same question for `genre` — the column the Retrieve_by_Genre node actually filters on
await q("""
    SELECT genre, count(*) AS n
    FROM books
    GROUP BY genre
    ORDER BY n DESC
""")

,genre,n
0,Fiction,2808
1,Nonfiction,1942
2,Children's Fiction,390
3,Children's Nonfiction,57


In [5]:
# column coverage — which fields are sparse enough to break a retrieval node?
await q("""
    SELECT
        count(*)                                        AS total,
        count(*) FILTER (WHERE description IS NULL)     AS no_description,
        count(*) FILTER (WHERE categories  IS NULL)     AS no_categories,
        count(*) FILTER (WHERE genre       IS NULL)     AS no_genre,
        count(*) FILTER (WHERE authors     IS NULL)     AS no_authors,
        count(*) FILTER (WHERE published_year IS NULL)  AS no_year,
        count(*) FILTER (WHERE embedding   IS NULL)     AS no_embedding
    FROM books
""")

,total,no_description,no_categories,no_genre,no_authors,no_year,no_embedding
0,5197,0,0,0,0,0,0


## Findings log

`note()` appends a timestamped entry to `data/exploration_notes.md` — a plain
markdown file that survives kernel restarts, notebook clears, and git checkouts.
Run the cell below once, then call `note(...)` whenever a query tells you something
worth keeping.

In [6]:
from datetime import datetime

NOTES_FILE = FilesLocationConstants.DATA_DIR / "exploration_notes.md"


def note(title: str, body: str = "", sql: str = "") -> None:
    """Append a finding to the exploration log."""
    stamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    entry = [f"\n## {title}", f"_{stamp}_\n"]
    if body:
        entry.append(body.strip() + "\n")
    if sql:
        entry.append(f"```sql\n{sql.strip()}\n```\n")

    if not NOTES_FILE.exists():
        NOTES_FILE.write_text("# Data Exploration Notes\n")
    with NOTES_FILE.open("a") as f:
        f.write("\n".join(entry))
    print(f"logged -> {NOTES_FILE}")


def notes() -> None:
    """Print everything logged so far."""
    print(NOTES_FILE.read_text() if NOTES_FILE.exists() else "(no notes yet)")

In [7]:
note(
    "categories is high-cardinality free text",
    f"{len(cats)} distinct values across {cats['n'].sum()} books; "
    f"top value '{cats.iloc[0]['categories']}' covers {cats.iloc[0]['n']} of them. "
    "Not a clean facet to filter on as-is.",
    sql="SELECT categories, count(*) FROM books GROUP BY categories ORDER BY 2 DESC",
)

logged -> /home/tuani/Book-Recommender/backend/data/exploration_notes.md


In [8]:
notes()

# Data Exploration Notes

## categories is high-cardinality free text
_2026-07-21 07:14_

480 distinct values across 5197 books; top value 'Fiction' covers 2111 of them. Not a clean facet to filter on as-is.

```sql
SELECT categories, count(*) FROM books GROUP BY categories ORDER BY 2 DESC
```

